# C · Archivos y memoria

**📄 PÁGINA · Archivos y memoria** · https://rayalucaria.org/fdd_o26/python/stack/archivos/

**📓 NOTEBOOK · c_archivos.ipynb · celdas C.0–C.5**

¿Te perdiste? Corre C.0 y salta a la celda que vamos.

Cada celda trae arriba qué compara y qué mirar, el código ya escrito y, debajo, «Deberías ver» con la salida de referencia. Córrelas con Shift+Enter y compara. La página que acompaña a este notebook es **Archivos y memoria** (sección 9.3 del curso).

- Todo lo que este notebook escribe va a `datos/`, que git ignora: no termina en tu entrega.
- Antes de entregar: *Clear All Outputs* y guarda. La revisión automática rechaza un notebook con salidas.

## C.0 · Preparación

Corre esta celda primero, y otra vez si reinicias el kernel. Es idempotente: correrla dos veces da lo mismo.

| Qué hace | Para qué |
|---|---|
| Importa todo lo que usan C.1–C.5 | Puedes saltar a cualquier celda después de ésta |
| Elige `N` (las filas de la tabla de ventas) | Deja **una** línea sin `#` |
| Imprime tu RAM y el `N` que te recomendamos | No cambia `N` sola: lo cambias tú |
| `datos.genera(N)` | Escribe `datos/ventas_{N}.csv` y `.parquet`; si ya existen (de `b_tablas.ipynb`), los reusa |
| `df = pl.read_parquet(...)` | `df` es **el original**: C.1 compara contra él lo que vuelve de cada archivo |

- `N_FILAS = min(N, 200_000)`: las carreras con objetos de Python (C.5) usan a lo más 200 mil filas, para que ninguna celda tarde de más.
- `df.schema` es el contrato de la tabla: el nombre y el tipo de cada columna. `Date` es una fecha sin hora; `Int64`, un entero; `String`, texto; `Float64`, un número con decimales.

In [ ]:
import pickle
import subprocess
import sys
import tracemalloc
from itertools import islice

import pandas as pd
import polars as pl

import datos
from datos import cronometra

# Elige el tamaño según tu máquina: deja UNA línea sin #.
N = 1_000_000        # por defecto: cualquier laptop
# N = 100_000        # 8 GB de RAM o menos, o si una celda tarda más de 30 s
# N = 10_000_000     # 16 GB o más; hasta ~3 GB en disco en datos/

print("Python:", sys.executable)
print("polars", pl.__version__, "· pandas", pd.__version__)
datos.recomienda_n()
rutas = datos.genera(N)
df = pl.read_parquet(rutas["parquet"])
N_FILAS = min(N, 200_000)  # para las carreras de objetos de Python (C.5)
CARPETA = datos.CARPETA
print(df.schema)

**Deberías ver** (N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB; tus números serán otros, lo que se mantiene es el patrón):

```text
Python: …/09_python/stack/.venv/bin/python
polars 2.0.0 · pandas 3.0.6
Tienes 31 GB de RAM → te recomendamos N = 1_000_000
Con más de 16 GB también puedes probar N = 10_000_000.
…/09_python/stack/datos/ventas_1000000.csv  1_000_000 filas  32.4 MB
…/09_python/stack/datos/ventas_1000000.parquet  1_000_000 filas  6.0 MB
Schema({'fecha': Date, 'tienda': String, 'producto': String, 'cantidad': Int64, 'precio': Float64})
```

- La primera línea termina en `.venv/bin/python` o `python3`. Si no, el kernel no es el del `.venv`: elígelo arriba a la derecha.
- Las dos rutas terminan en `ventas_1000000` (o en tu `N`): el archivo es el de tu `N`.
- Guarda en la cabeza el `schema`: cinco columnas, cinco tipos. C.1 y C.2 preguntan cuáles sobreviven a un viaje por disco.

## C.1 · Carrera de formatos: cuánto ocupa, cuánto tarda y qué tipos regresan

La misma tabla `df` (`N` filas) se escribe en seis formatos dentro de `datos/` y se vuelve a leer de tres maneras.

| Formato | Qué es |
|---|---|
| **CSV** | Texto: una fila por línea, valores separados por comas. No guarda tipos |
| **NDJSON** | Texto: un objeto JSON por línea (`{"fecha":"2025-03-01",…}`). Tampoco guarda tipos de fecha |
| **Parquet** | Binario, **por columnas**: guarda junta cada columna, comprimida, con su tipo |
| snappy / zstd | Dos compresores de Parquet: snappy comprime menos y más rápido; zstd comprime más (es el que Polars usa si no le dices) |
| **Feather** | Arrow IPC: las columnas tal como están en memoria; sin comprimir por omisión |
| **pickle** | El formato de Python para guardar *cualquier* objeto, no sólo tablas |

| Columna de la tabla | Qué mide |
|---|---|
| `MB` | Tamaño del archivo en disco |
| `escribir_s`, `leer_todo_s` | Mejor de 2 corridas (`cronometra(..., veces=2)`), en segundos |
| `leer_2_columnas_s` | Leer sólo `tienda` y `precio` |
| `tipos_intactos` | `True` si el `schema` que vuelve es **igual** al de `df` |

**Qué mirar:**

- La columna `MB`, de arriba abajo: la tabla sale ordenada por tamaño.
- `leer_2_columnas_s` contra `leer_todo_s`: un formato por columnas lee sólo las columnas que pides; uno de texto tiene que recorrer cada línea entera, y pickle, cargar el objeto completo.
- `tipos_intactos` en los dos formatos de texto.

**La particularidad:** pickle sale bien en esta carrera: tipos intactos, tamaño y tiempo razonables. Polars guarda un DataFrame en pickle usando Arrow por dentro, por eso pesa lo mismo que Feather. **Su problema no es la velocidad: es C.4.**

Se usan 2 corridas y no 3 para que la celda no pase de ~15 s con `N = 1_000_000` (en la máquina de referencia tarda ~4 s). Con `N = 10_000_000` tarda ~50 s y deja ~2.4 GB más en `datos/`; bórralos al terminar con `rm datos/c1*`.

In [ ]:
def mb(ruta):
    return round(ruta.stat().st_size / 1e6, 1)


def escribe_pickle(ruta):
    with open(ruta, "wb") as f:
        pickle.dump(df, f)


def lee_pickle(ruta):
    with open(ruta, "rb") as f:
        return pickle.load(f)


# formato: (archivo, escribir, leer todo, leer sólo 2 columnas)
DOS = ["tienda", "precio"]
FORMATOS = {
    "CSV": ("c1.csv", df.write_csv, pl.read_csv,
            lambda r: pl.read_csv(r, columns=DOS)),
    "NDJSON": ("c1.ndjson", df.write_ndjson, pl.read_ndjson,
               lambda r: pl.scan_ndjson(r).select(DOS).collect()),
    "Parquet snappy": ("c1_snappy.parquet",
                       lambda r: df.write_parquet(r, compression="snappy"),
                       pl.read_parquet, lambda r: pl.read_parquet(r, columns=DOS)),
    "Parquet zstd": ("c1_zstd.parquet", df.write_parquet, pl.read_parquet,
                     lambda r: pl.read_parquet(r, columns=DOS)),
    "Feather": ("c1.feather", df.write_ipc, pl.read_ipc,
                lambda r: pl.read_ipc(r, columns=DOS)),
    "pickle": ("c1.pkl", escribe_pickle, lee_pickle,
               lambda r: lee_pickle(r).select(DOS)),
}


def corre(formato, archivo, escribe, lee, lee_dos):
    """Una fila de la tabla: escribe, relee y compara el schema con el de df."""
    ruta = CARPETA / archivo
    return {
        "formato": formato,
        "escribir_s": round(cronometra(lambda: escribe(ruta), veces=2), 3),
        "MB": mb(ruta),
        "leer_todo_s": round(cronometra(lambda: lee(ruta), veces=2), 3),
        "leer_2_columnas_s": round(cronometra(lambda: lee_dos(ruta), veces=2), 3),
        "tipos_intactos": lee(ruta).schema == df.schema,
    }


filas = [corre(formato, *partes) for formato, partes in FORMATOS.items()]
pl.DataFrame(filas).select("formato", "MB", pl.exclude("formato", "MB")).sort("MB")

**Deberías ver** (N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB; tus números serán otros, lo que se mantiene es el patrón):

```text
shape: (6, 6)
┌────────────────┬──────┬────────────┬─────────────┬───────────────────┬────────────────┐
│ formato        ┆ MB   ┆ escribir_s ┆ leer_todo_s ┆ leer_2_columnas_s ┆ tipos_intactos │
│ ---            ┆ ---  ┆ ---        ┆ ---         ┆ ---               ┆ ---            │
│ str            ┆ f64  ┆ f64        ┆ f64         ┆ f64               ┆ bool           │
╞════════════════╪══════╪════════════╪═════════════╪═══════════════════╪════════════════╡
│ Parquet zstd   ┆ 6.0  ┆ 0.067      ┆ 0.011       ┆ 0.008             ┆ true           │
│ Parquet snappy ┆ 7.6  ┆ 0.056      ┆ 0.009       ┆ 0.007             ┆ true           │
│ CSV            ┆ 32.4 ┆ 0.138      ┆ 0.067       ┆ 0.043             ┆ false          │
│ Feather        ┆ 52.1 ┆ 0.127      ┆ 0.011       ┆ 0.008             ┆ true           │
│ pickle         ┆ 52.1 ┆ 0.238      ┆ 0.081       ┆ 0.078             ┆ true           │
│ NDJSON         ┆ 88.5 ┆ 0.357      ┆ 0.233       ┆ 0.187             ┆ false          │
└────────────────┴──────┴────────────┴─────────────┴───────────────────┴────────────────┘
```

- **Columnas + compresión**: Parquet zstd ocupa ~5 veces menos que el CSV con los mismos datos y ~15 veces menos que NDJSON. Este orden de tamaños se mantiene en cualquier máquina.
- **Los formatos de texto pierden tipos**: CSV y NDJSON dan `false`, porque la fecha vuelve como `String`. C.2 lo muestra de cerca.
- Parquet y Feather leen 6–8 veces más rápido que el CSV y ~20 veces más que NDJSON; **entre ellos casi empatan** (su orden cambia entre corridas). pickle pesa lo mismo que Feather y lee varias veces más lento.
- Con `N = 1_000_000`, leer 2 columnas o todas cuesta milisegundos y la diferencia se pierde en el ruido. Con `N = 10_000_000` se ve: Parquet zstd baja de 0.165 a 0.099 s y CSV de 0.949 a 0.487 s; pickle no baja (0.847 → 0.907 s), porque tiene que cargar el objeto entero.

## C.2 · Una fecha y un entero con vacío, de ida y vuelta

Dos filas: una fecha (`Date`) y una cantidad entera con un vacío (`None`, que Polars guarda como `null`). Se escriben en CSV y en Parquet, y cada archivo se relee con Polars y con pandas.

**Qué mirar:** el tipo de `fecha` y de `cantidad` en cada renglón, contra el `original`.

| Término | Qué es |
|---|---|
| `null` (Polars) | «No hay dato», en cualquier tipo de columna, sin cambiar el tipo |
| `float64` (pandas) | Número con decimales. pandas lo usa para una columna de enteros con un vacío, porque su vacío por omisión, `NaN`, es un `float` |
| `object` (pandas) | Una columna de objetos de Python sueltos; pandas no tiene un tipo de «sólo fecha» |

**La particularidad:** el CSV no guarda ningún tipo; quien lo lee **adivina**. Polars y pandas adivinan distinto con el mismo archivo.

In [ ]:
from datetime import date

chico = pl.DataFrame({"fecha": [date(2025, 3, 1), date(2025, 3, 2)],
                      "cantidad": [3, None]})
chico.write_csv(CARPETA / "c2.csv")
chico.write_parquet(CARPETA / "c2.parquet")

print("original        ", dict(chico.schema))
print("CSV · Polars    ", dict(pl.read_csv(CARPETA / "c2.csv").schema))
print("CSV · pandas    ", pd.read_csv(CARPETA / "c2.csv").dtypes.to_dict())
print("Parquet · Polars", dict(pl.read_parquet(CARPETA / "c2.parquet").schema))
print("Parquet · pandas", pd.read_parquet(CARPETA / "c2.parquet").dtypes.to_dict())

**Deberías ver:**

```text
original         {'fecha': Date, 'cantidad': Int64}
CSV · Polars     {'fecha': String, 'cantidad': Int64}
CSV · pandas     {'fecha': <StringDtype(na_value=nan)>, 'cantidad': dtype('float64')}
Parquet · Polars {'fecha': Date, 'cantidad': Int64}
Parquet · pandas {'fecha': dtype('O'), 'cantidad': dtype('float64')}
```

- **El CSV pierde la fecha** con los dos lectores: vuelve como texto (`String` en Polars, `StringDtype` en pandas).
- pandas además convierte la cantidad en `float64`: el `3` vuelve como `3.0`. Es lo mismo que viste en B.7.
- **Parquet conserva los tipos** con Polars. Con pandas la fecha llega como `object` (objetos `date` de Python) y la cantidad como `float64`: pandas no tiene esos tipos por omisión, no es culpa del archivo.

**📄 PÁGINA · Archivos y memoria · sección «Parquet ocupa menos y la fecha vuelve como fecha»**

## C.3 · `with` cierra el archivo aunque haya un error

Un archivo abierto ocupa un recurso del sistema hasta que lo cierras. **Cerrarlo** es lo que garantiza que lo escrito quedó en disco.

- `with open(ruta, "w", encoding="utf-8") as f:` abre el archivo y lo nombra `f`. Al salir del bloque —por el final o **por un error**— Python llama a `f.close()` por ti.
- `raise ValueError(...)` provoca un error a la mitad, a propósito. `try`/`except` (lo viste en DataCamp) lo atrapa para que la celda siga.

**Qué mirar:** `f.closed` después del error, y qué quedó escrito en `datos/c3.txt`.

In [ ]:
ruta = CARPETA / "c3.txt"
try:
    with open(ruta, "w", encoding="utf-8") as f:
        f.write("la primera línea sí se escribió\n")
        raise ValueError("algo falló a la mitad")
except ValueError as e:
    print("error:", e)

print("¿cerrado?", f.closed)
print(ruta.read_text(encoding="utf-8"), end="")

**Deberías ver:**

```text
error: algo falló a la mitad
¿cerrado? True
la primera línea sí se escribió
```

- `¿cerrado? True`: el error salió del bloque `with` y aun así el archivo quedó cerrado.
- La línea escrita antes del error está en el archivo: cerrar guardó lo pendiente.
- Sin `with` (`f = open(...)` y un error antes de `f.close()`) el archivo queda abierto hasta que Python lo recoja, y lo escrito puede no haber llegado al disco.

**📄 PÁGINA · Archivos y memoria · sección «with cierra el archivo aunque haya un error»**

## C.4 · Cargar un pickle ejecuta código

C.1 mostró que pickle guarda bien una tabla. El problema es **cómo** la reconstruye al cargarla.

- `__reduce__` es el método que pickle le pide a un objeto al **guardarlo**: «¿cómo te reconstruyo?». Regresa una pareja `(función, argumentos)`.
- `pickle.dumps(obj)` convierte el objeto en bytes; esos bytes guardan **el nombre** de la función y sus argumentos.
- `pickle.loads(bytes)` reconstruye: **llama a esa función** con esos argumentos.

Aquí la función es `print`, con un texto: **el ejemplo es inofensivo a propósito**. Nada obliga a que sea `print`.

**Qué mirar:** en los bytes, las palabras `builtins` y `print`; y que el mensaje aparece al **cargar**, sin que ninguna línea de la celda llame a `print` con ese texto.

In [ ]:
class Inofensivo:
    def __reduce__(self):
        # Le dice a pickle: «para reconstruirme, llama a print con este texto».
        return (print, ("Esto lo ejecutó pickle.loads, no tu código.",))


contenido = pickle.dumps(Inofensivo())
print("bytes guardados:", contenido)
resultado = pickle.loads(contenido)
print("pickle.loads regresó:", resultado)

**Deberías ver:**

```text
bytes guardados: b'\x80\x05\x95J\x00\x00\x00\x00\x00\x00\x00\x8c\x08builtins\x94\x8c\x05print\x94\x93\x94\x8c-Esto lo ejecut\xc3\xb3 pickle.loads, no tu c\xc3\xb3digo.\x94\x85\x94R\x94.'
Esto lo ejecutó pickle.loads, no tu código.
pickle.loads regresó: None
```

- **`pickle.loads` llamó a `print`.** Con un pickle ajeno, esa función podría ser `os.system` con cualquier comando: borrar archivos, mandar tus llaves a otro servidor. Pasa en cuanto lo cargas, antes de que veas los datos.
- Los bytes nombran `builtins` y `print` como texto: para cargarlo no hace falta que la clase `Inofensivo` exista en quien lo abre.
- Lo mismo vale para todo lo que usa pickle por dentro, como `pd.read_pickle`. **Carga pickles sólo tuyos**; para compartir una tabla, Parquet: guarda datos, no instrucciones.

**📄 PÁGINA · Archivos y memoria · sección «Cargar un pickle ejecuta código»**

## C.5 · Todo a la vez vs uno a la vez

La tarea: sumar la columna `precio` del CSV. Tres maneras.

| Opción | Qué hace |
|---|---|
| `lista` | Lee **todas** las líneas a una lista y luego suma |
| `generador` | Una función con `yield` entrega **un** precio, se pausa, y sigue cuando le piden el siguiente: en memoria vive una línea a la vez |
| `scan` | `pl.scan_csv(...).select(pl.col("precio").sum()).collect()`: Polars lee el archivo en Rust, por bloques y con varios hilos |

`yield` convierte una función en un **generador**: al llamarla no corre nada; cada vuelta del `for` (o de `sum`) la avanza hasta el siguiente `yield`. `islice(f, n)` toma sólo las primeras `n` líneas del archivo.

**Dos mediciones, por dos motivos:**

1. **Dentro de este kernel**, con `N_FILAS` filas: `segundos` en una pasada **sin** `tracemalloc`, y `pico_tracemalloc_mb` en otra pasada **con** él. `tracemalloc` registra cada memoria que pide Python y eso hace todo varias veces más lento: por eso tiempo y memoria van en pasadas separadas.
2. **El archivo entero** (`N` filas), cada opción en un proceso nuevo con `mide.py`. `tracemalloc` no ve la memoria que Polars pide en Rust; el **RSS** (la memoria que el sistema le dio al proceso entero; `htop` la muestra en RES) sí. `delta_mb` es lo que sumó la tarea sobre la memoria con la librería ya cargada (`rss_base_mb`).

**Qué mirar:** la columna de memoria de cada tabla, y cómo cambiaría si `N` fuera 10 veces más grande.

In [ ]:
csv = rutas["csv"]


def precios_en_lista(n):
    """Todo a la vez: las n líneas entran a una lista y luego se suman."""
    with open(csv, encoding="utf-8") as f:
        next(f)  # salta el encabezado
        lineas = list(islice(f, n))
    return sum(float(linea.rsplit(",", 1)[1]) for linea in lineas)


def precios_uno_a_uno(n):
    """Uno a la vez: un generador entrega cada precio y lo olvida."""
    with open(csv, encoding="utf-8") as f:
        next(f)
        for linea in islice(f, n):
            yield float(linea.rsplit(",", 1)[1])


def pico_mb(fn):
    """Memoria pico que pidió Python mientras corría fn, en MB."""
    tracemalloc.start()
    fn()
    pico = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return round(pico / 1e6, 2)


opciones = {
    "lista": lambda: precios_en_lista(N_FILAS),
    "generador": lambda: sum(precios_uno_a_uno(N_FILAS)),
}
# Pasada 1, el tiempo, sin tracemalloc (tracemalloc hace todo más lento).
tiempos = {k: round(cronometra(fn), 3) for k, fn in opciones.items()}
# Pasada 2, la memoria, con tracemalloc.
picos = {k: pico_mb(fn) for k, fn in opciones.items()}
print(f"Con N_FILAS = {N_FILAS:_} filas, dentro de este kernel:")
print(pl.DataFrame({"opcion": list(opciones), "segundos": list(tiempos.values()),
                    "pico_tracemalloc_mb": list(picos.values())}))

# El archivo entero (N filas), cada opción en un proceso nuevo, con RSS.
print(f"Con N = {N:_} filas, un proceso nuevo por opción (mide.py):")
lineas = []
for escenario in ["lista", "generador", "scan"]:
    salida = subprocess.run([sys.executable, "mide.py", escenario, str(N)],
                            capture_output=True, text=True, check=True)
    lineas.append(salida.stdout.strip().split("\t"))
columnas = ["escenario", "segundos", "rss_base_mb", "rss_pico_mb", "delta_mb"]
pl.DataFrame(lineas, orient="row", schema=columnas).with_columns(
    pl.exclude("escenario").cast(pl.Float64))

**Deberías ver** (N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB; tus números serán otros, lo que se mantiene es el patrón):

```text
Con N_FILAS = 200_000 filas, dentro de este kernel:
shape: (2, 3)
┌───────────┬──────────┬─────────────────────┐
│ opcion    ┆ segundos ┆ pico_tracemalloc_mb │
│ ---       ┆ ---      ┆ ---                 │
│ str       ┆ f64      ┆ f64                 │
╞═══════════╪══════════╪═════════════════════╡
│ lista     ┆ 0.106    ┆ 16.83               │
│ generador ┆ 0.098    ┆ 0.17                │
└───────────┴──────────┴─────────────────────┘
Con N = 1_000_000 filas, un proceso nuevo por opción (mide.py):
shape: (3, 5)
┌───────────┬──────────┬─────────────┬─────────────┬──────────┐
│ escenario ┆ segundos ┆ rss_base_mb ┆ rss_pico_mb ┆ delta_mb │
│ ---       ┆ ---      ┆ ---         ┆ ---         ┆ ---      │
│ str       ┆ f64      ┆ f64         ┆ f64         ┆ f64      │
╞═══════════╪══════════╪═════════════╪═════════════╪══════════╡
│ lista     ┆ 0.462    ┆ 15.6        ┆ 109.5       ┆ 93.9     │
│ generador ┆ 0.389    ┆ 15.6        ┆ 15.8        ┆ 0.1      │
│ scan      ┆ 0.043    ┆ 46.5        ┆ 123.7       ┆ 77.2     │
└───────────┴──────────┴─────────────┴─────────────┴──────────┘
```

- **La lista guarda el archivo entero; el generador, una línea**: 16.83 MB contra 0.17 MB en `tracemalloc`, y 93.9 MB contra 0.1 MB de `delta_mb`. Con `N = 10_000_000`, la lista sube a 937.7 MB de delta y el generador se queda en 0.1.
- En tiempo, `lista` y `generador` **casi empatan**: los dos convierten cada línea en Python.
- **`scan` es ~10 veces más rápido, pero no es «uno a la vez»**: lee bloques grandes en varios hilos y su `delta_mb` también crece con `N` (506.4 MB con 10 millones). Si lo que te falta es memoria, el generador es el único que no crece.

**📄 Regresa a la página Archivos y memoria.** Antes, *Restart* o *Shutdown* de este kernel: libera la memoria para el siguiente notebook.